<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #6a11cb 0%, #2575fc 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎬 LTX-2.5 22B Distilled - Dual GPU Video Generator</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Kaggle Dual T4 GPU Edition - Created by <strong>AIQUEST Academy</strong></h3>
  <p style='color: #ddd; margin: 0; text-align: center;'>Official LTX-2.5 22B Distilled • Persistent In-Memory Server • 8-Step Ultra Fast Denoising • Synchronized Stereo Audio • Dual GPU T4 x2</p>
</div>

---

<div align="center">
  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Kaggle-GPU%20T4%20x2-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />
  <br>
  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
</div>

---

### What is LTX-2.5 22B Distilled?

**LTX-2.5** is the latest major foundation video & audio model release from Lightricks, delivering significantly improved motion dynamics, sharper photorealism, and native stereo sound.

| Feature | Details |
|---|---|
| **Model** | `LTX-2.5 22B Distilled` (`ltx2_25_22B_distilled` profile) |
| **Weights** | `ltx-2.5-22b-distilled_diffusion_model_int8_convrot.safetensors` (~19.5 GB) |
| **Quantization** | Native INT8 Convrot (Zero fallback latency on Turing `sm_75` Tensor Cores) |
| **Accelerator** | Kaggle Dual GPU (`GPU T4 x2` - 32GB Total VRAM across `cuda:0` & `cuda:1`) |
| **Text Encoder** | `gemma4-12b-ltx-v1` (INT8 Convrot text encoder) |
| **Execution Mode** | **Persistent In-Memory Server** (Loaded ONCE before Gradio URL; 0s load time per generation) |
| **Inference Speed** | 8 Denoising Steps (Single-Stage Turbo Pipeline) |
| **UI** | **Custom AIQUEST Academy Web Interface** (Real-Time WebSocket Streaming & Timeout-Free) |

### Quick Start Guide
1. **Settings → Accelerator → GPU T4 x2**
2. **Turn on Internet** in the Kaggle sidebar menu
3. Run all cells in order
4. Open the Gradio public link to generate videos with live step-by-step streaming!

In [ ]:
#@title Step 1: Environment Setup, Dependencies & Dual GPU Optimization (GPU T4 x2)
import os
import sys
import gc
import psutil
import torch
import subprocess

print('=== Kaggle Dual GPU Environment Setup (GPU T4 x2) ===')
ram = psutil.virtual_memory()
print(f'System RAM: {ram.total / 1024**3:.1f} GB total, {ram.available / 1024**3:.1f} GB available')

gpu_count = torch.cuda.device_count()
print(f'Detected GPUs: {gpu_count}')
for i in range(gpu_count):
    props = torch.cuda.get_device_properties(i)
    free_vram = torch.cuda.mem_get_info(i)[0] / 1024**3
    total_vram = props.total_memory / 1024**3
    print(f'  - cuda:{i} ({props.name}): {free_vram:.1f} GB free / {total_vram:.1f} GB total (Capability {props.major}.{props.minor})')

if gpu_count < 2:
    print('\n⚠️ WARNING: Less than 2 GPUs detected! Please verify Settings → Accelerator → GPU T4 x2')
else:
    print('\n✅ Dual GPU Configuration (GPU T4 x2) Verified!')

# System cache clearing and overcommit settings
os.system('rm -rf /kaggle/tmp/* /tmp/*')
os.system('echo 3 | sudo tee /proc/sys/vm/drop_caches > /dev/null 2>&1')
os.system('echo 1 | sudo tee /proc/sys/vm/overcommit_memory > /dev/null 2>&1')
gc.collect()

# PyTorch allocator & tokenizer memory optimizations
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,garbage_collection_threshold:0.6'
os.environ['MALLOC_TRIM_THRESHOLD_'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TORCH_CUDNN_V8_API_ENABLED'] = '1'

# Clone or clean Wan2GP framework with LTX-2.5 support
if not os.path.exists('Wan2GP'):
    print('Cloning latest Wan2GP repository...')
    subprocess.run(['git', 'clone', 'https://github.com/DeepBeepMeep/Wan2GP.git'], check=True)
    print('✓ Wan2GP cloned successfully!')
else:
    print('✓ Wan2GP repository exists. Resetting to clean git state...')
    subprocess.run(['git', 'checkout', '.'], cwd='Wan2GP', check=False)
    subprocess.run(['git', 'pull', 'origin', 'main'], cwd='Wan2GP', check=False)

# Install complete Wan2GP dependencies
print('Installing all core Wan2GP dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'sageattention'], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--timeout', '180',
    '-r', 'Wan2GP/requirements.txt'
], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'openai-whisper',
    'ffmpeg-python',
    'onnxruntime',
    'rembg',
    'gradio>=4.40.0',
    'bitsandbytes',
    'soundfile',
    'opencv-python'
], check=False)

# Apply automatic high-speed and visual quality patches to Wan2GP
print('Applying high-speed FP16 & crystal-clear quality patches to Wan2GP...')
for base_dir in ['Wan2GP', '.']:
    # 1. High-Speed FP16 on T4 Tensor Cores (~24s/step)
    distilled_file = os.path.join(base_dir, 'models', 'ltx2', 'ltx_pipelines', 'distilled.py')
    if os.path.isfile(distilled_file):
        with open(distilled_file, 'r', encoding='utf-8') as f:
            dist_code = f.read()
        dist_code = dist_code.replace("self.dtype = torch.bfloat16", "self.dtype = getattr(models, 'dtype', torch.float16)")
        dist_code = dist_code.replace("dtype = torch.bfloat16", "dtype = getattr(self, 'dtype', torch.float16)")
        with open(distilled_file, 'w', encoding='utf-8') as f:
            f.write(dist_code)

    # 2. PixelNorm Float32 precision (fixes FP16 overflow to solid gray background)
    norm_file = os.path.join(base_dir, 'models', 'ltx2', 'ltx_core', 'model', 'common', 'normalization.py')
    if os.path.isfile(norm_file):
        with open(norm_file, 'r', encoding='utf-8') as f:
            norm_code = f.read()
        if 'x_float = x.float()' not in norm_code:
            norm_code = norm_code.replace(
                "        mean_sq = torch.mean(x**2, dim=self.dim, keepdim=True)\n        # Normalize by the root-mean-square (RMS).\n        rms = torch.sqrt(mean_sq + self.eps)\n        return x / rms",
                "        orig_dtype = x.dtype\n        x_float = x.float()\n        mean_sq = torch.mean(x_float**2, dim=self.dim, keepdim=True)\n        rms = torch.sqrt(mean_sq + self.eps)\n        return (x_float / rms).to(orig_dtype)"
            )
            with open(norm_file, 'w', encoding='utf-8') as f:
                f.write(norm_code)

    # 3. Gemma Feature Extractor RMS float32 precision
    fe_file = os.path.join(base_dir, 'models', 'ltx2', 'ltx_core', 'text_encoders', 'gemma', 'feature_extractor.py')
    if os.path.isfile(fe_file):
        with open(fe_file, 'r', encoding='utf-8') as f:
            fe_code = f.read()
        if 'enc_f = encoded_text.float()' not in fe_code:
            fe_code = fe_code.replace(
                "def _norm_and_concat_per_token_rms(encoded_text: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n    b, t, d, l = encoded_text.shape\n    variance = torch.mean(encoded_text**2, dim=2, keepdim=True)\n    normed = encoded_text * torch.rsqrt(variance + 1e-6)\n    normed = normed.reshape(b, t, d * l)\n    mask_3d = attention_mask.bool().unsqueeze(-1)\n    return torch.where(mask_3d, normed, torch.zeros_like(normed))",
                "def _norm_and_concat_per_token_rms(encoded_text: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n    orig_dtype = encoded_text.dtype\n    enc_f = encoded_text.float()\n    b, t, d, l = enc_f.shape\n    variance = torch.mean(enc_f**2, dim=2, keepdim=True)\n    normed = enc_f * torch.rsqrt(variance + 1e-6)\n    normed = normed.reshape(b, t, d * l).to(orig_dtype)\n    mask_3d = attention_mask.bool().unsqueeze(-1)\n    return torch.where(mask_3d, normed, torch.zeros_like(normed))"
            )
            with open(fe_file, 'w', encoding='utf-8') as f:
                f.write(fe_code)

    # 4. VAE CausalConv3d auto-cast
    conv_file = os.path.join(base_dir, 'models', 'ltx2', 'ltx_core', 'model', 'video_vae', 'convolution.py')
    if os.path.isfile(conv_file):
        with open(conv_file, 'r', encoding='utf-8') as f:
            conv_code = f.read()
        if 'x = self.conv(x)' in conv_code and 'x.dtype != self.conv.weight.dtype' not in conv_code:
            conv_code = conv_code.replace(
                "        x = self.conv(x)",
                "        if hasattr(self, 'conv') and hasattr(self.conv, 'weight') and x.dtype != self.conv.weight.dtype:\n            x = x.to(dtype=self.conv.weight.dtype)\n        x = self.conv(x)"
            )
            with open(conv_file, 'w', encoding='utf-8') as f:
                f.write(conv_code)

    # 5. Audio VAE & Vocoder buffer auto-casts
    audio_vae_file = os.path.join(base_dir, 'models', 'ltx2', 'ltx_core', 'model', 'audio_vae', 'audio_vae.py')
    if os.path.isfile(audio_vae_file):
        with open(audio_vae_file, 'r', encoding='utf-8') as f:
            audio_vae_code = f.read()
        if 'decoded_audio = audio_decoder(latent)' in audio_vae_code and 'latent.to(dtype=' not in audio_vae_code:
            audio_vae_code = audio_vae_code.replace(
                "    decoded_audio = audio_decoder(latent)",
                "    if hasattr(audio_decoder, 'conv_in') and hasattr(audio_decoder.conv_in, 'weight'):\n        latent = latent.to(dtype=audio_decoder.conv_in.weight.dtype)\n    decoded_audio = audio_decoder(latent)\n    if hasattr(vocoder, 'vocoder') and hasattr(vocoder.vocoder, 'conv_pre'):\n        decoded_audio = decoded_audio.to(dtype=vocoder.vocoder.conv_pre.weight.dtype)"
            )
            with open(audio_vae_file, 'w', encoding='utf-8') as f:
                f.write(audio_vae_code)

    vocoder_file = os.path.join(base_dir, 'models', 'ltx2', 'ltx_core', 'model', 'audio_vae', 'vocoder.py')
    if os.path.isfile(vocoder_file):
        with open(vocoder_file, 'r', encoding='utf-8') as f:
            vocoder_code = f.read()
        vocoder_code = vocoder_code.replace(
            "return F.conv1d(x, self.filter.expand(n_channels, -1, -1), stride=self.stride, groups=n_channels)",
            "filt = self.filter.to(dtype=x.dtype, device=x.device).expand(n_channels, -1, -1)\n        return F.conv1d(x, filt, stride=self.stride, groups=n_channels)"
        )
        vocoder_code = vocoder_code.replace(
            "spec = F.conv1d(y, self.forward_basis, stride=self.hop_length, padding=0)",
            "basis = self.forward_basis.to(dtype=y.dtype, device=y.device)\n        spec = F.conv1d(y, basis, stride=self.hop_length, padding=0)"
        )
        with open(vocoder_file, 'w', encoding='utf-8') as f:
            f.write(vocoder_code)

print('✅ Environment, Dependencies & Quality Patches Complete!')

In [ ]:
#@title Step 2: Download Official LTX-2.5 22B Distilled Weights & Text Encoder
import os
import sys
import time
import json
import shutil

# Ensure clean huggingface_hub module state
for mod in list(sys.modules.keys()):
    if mod.startswith('huggingface_hub'):
        del sys.modules[mod]

from huggingface_hub import hf_hub_download

print('=== Downloading Official LTX-2.5 Distilled Weights ===')

os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

TMP_DIR = '/tmp/models'
MODEL_DIR = os.path.abspath('Wan2GP/models')
CKPTS_DIR = os.path.abspath('Wan2GP/ckpts')
OUTPUTS_DIR = os.path.abspath('Wan2GP/outputs')
os.makedirs(TMP_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CKPTS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)
os.makedirs('/kaggle/working/outputs', exist_ok=True)

def safe_symlink(src, dest):
    if not os.path.exists(dest):
        try:
            os.symlink(src, dest)
        except Exception:
            pass

LTX25_REPO = 'DeepBeepMeep/LTX-2'

# 1. Official LTX-2.5 22B Distilled Diffusion Model (INT8 Convrot, ~19.5 GB) -> Download to /tmp/models
LTX25_DISTILLED_FILE = 'ltx-2.5-22b-distilled_diffusion_model_int8_convrot.safetensors'
dest_tf = os.path.join(MODEL_DIR, LTX25_DISTILLED_FILE)
dest_tf_ckpt = os.path.join(CKPTS_DIR, LTX25_DISTILLED_FILE)

if os.path.exists(dest_tf) and os.path.exists(dest_tf_ckpt):
    print(f'  ✓ Already exists: {LTX25_DISTILLED_FILE}')
else:
    print(f'Downloading LTX-2.5 Distilled INT8 weights ({LTX25_DISTILLED_FILE}, ~19.5 GB to temp storage)...')
    sys.stdout.flush()
    t0 = time.time()
    hf_hub_download(repo_id=LTX25_REPO, filename=LTX25_DISTILLED_FILE, local_dir=TMP_DIR)
    actual = os.path.join(TMP_DIR, LTX25_DISTILLED_FILE)
    shutil.rmtree(os.path.join(TMP_DIR, '.cache'), ignore_errors=True)
    if os.path.exists(actual):
        safe_symlink(actual, dest_tf)
        safe_symlink(actual, dest_tf_ckpt)
        elapsed = time.time() - t0
        print(f'  ✓ {LTX25_DISTILLED_FILE} ready! (Downloaded in {elapsed:.1f}s)')
        sys.stdout.flush()

# 2. Official LTX-2.5 Companion Components (Spatial Upscaler, VAEs, Connectors, Vocoder) -> Download to Wan2GP/models
LTX25_COMPANION_FILES = [
    'ltx-2.5-spatial-upscaler-x2-1.0_bf16.safetensors',
    'ltx-2.5-temporal-upscaler-x2-1.0_bf16.safetensors',
    'ltx-2.5-22b_video_embeddings_connector_int8_convrot.safetensors',
    'ltx-2.5-22b_audio_embeddings_connector_int8_convrot.safetensors',
    'ltx-2.5-22b_text_embedding_projection_bf16.safetensors',
    'ltx-2.5-22b_video_vae_bf16.safetensors',
    'ltx-2.5-22b_audio_vae_bf16.safetensors',
    'ltx-2.5-22b_vocoder_bf16.safetensors',
]

for f in LTX25_COMPANION_FILES:
    dest_m = os.path.join(MODEL_DIR, f)
    dest_c = os.path.join(CKPTS_DIR, f)
    if os.path.exists(dest_m) and os.path.exists(dest_c):
        print(f'  ✓ Already exists: {f}')
        continue
    print(f'Downloading {f}...')
    sys.stdout.flush()
    try:
        hf_hub_download(repo_id=LTX25_REPO, filename=f, local_dir=MODEL_DIR)
        shutil.rmtree(os.path.join(MODEL_DIR, '.cache'), ignore_errors=True)
        if os.path.exists(dest_m):
            safe_symlink(dest_m, dest_c)
            print(f'  ✓ {f} complete!')
    except Exception as e:
        print(f'  ⚠️ Warning downloading {f}: {e}')
    sys.stdout.flush()

# 3. LTX-2.5 Dedicated Gemma4 Text Encoder & Tokenizer Files (gemma4-12b-ltx-v1)
GEMMA4_SUBFOLDER = 'gemma4-12b-ltx-v1'
GEMMA4_DIR = os.path.join(MODEL_DIR, GEMMA4_SUBFOLDER)
GEMMA4_CKPT_DIR = os.path.join(CKPTS_DIR, GEMMA4_SUBFOLDER)
os.makedirs(GEMMA4_DIR, exist_ok=True)
os.makedirs(GEMMA4_CKPT_DIR, exist_ok=True)

GEMMA4_FILES = [
    'gemma4-12b-ltx-v1_int8_convrot.safetensors',
    'config.json',
    'chat_template.jinja',
    'tokenizer.json',
    'tokenizer_config.json'
]

print(f'Preparing LTX-2.5 Text Encoder ({GEMMA4_SUBFOLDER})...')
sys.stdout.flush()
for gf in GEMMA4_FILES:
    dest_gm = os.path.join(GEMMA4_DIR, gf)
    dest_gc = os.path.join(GEMMA4_CKPT_DIR, gf)
    if os.path.exists(dest_gm) and os.path.exists(dest_gc):
        continue
    print(f'Downloading {gf}...')
    sys.stdout.flush()
    try:
        hf_hub_download(
            repo_id=LTX25_REPO,
            subfolder=GEMMA4_SUBFOLDER,
            filename=gf,
            local_dir=MODEL_DIR
        )
        shutil.rmtree(os.path.join(MODEL_DIR, '.cache'), ignore_errors=True)
        # Handle nested subfolder if present
        nested = os.path.join(GEMMA4_DIR, GEMMA4_SUBFOLDER, gf)
        if os.path.exists(nested):
            shutil.move(nested, dest_gm)
        if os.path.exists(dest_gm):
            safe_symlink(dest_gm, dest_gc)
    except Exception as e:
        print(f'  Note on {gf}: {e}')
    sys.stdout.flush()

# Ensure all Gemma 4 files are symlinked to ckpts
for gf in GEMMA4_FILES:
    dest_gm = os.path.join(GEMMA4_DIR, gf)
    dest_gc = os.path.join(GEMMA4_CKPT_DIR, gf)
    if os.path.exists(dest_gm) and not os.path.exists(dest_gc):
        safe_symlink(dest_gm, dest_gc)

print('  ✓ LTX-2.5 Gemma-4 text encoder ready!')

# 4. Configure Wan2GP Defaults
DEFAULTS_DIR = os.path.abspath('Wan2GP/defaults')
distilled_default_config = {
    "model": {
        "name": "LTX-2 2.5 Distilled 22B",
        "architecture": "ltx2_25_22B",
        "description": "LTX-2.5 Distilled INT8 model optimized for fast 8-step single-stage inference on Dual GPUs.",
        "URLs": [
            "https://huggingface.co/DeepBeepMeep/LTX-2/resolve/main/ltx-2.5-22b-distilled_diffusion_model_bf16.safetensors",
            "https://huggingface.co/DeepBeepMeep/LTX-2/resolve/main/ltx-2.5-22b-distilled_diffusion_model_int8_convrot.safetensors"
        ],
        "preload_URLs": [],
        "ltx2_pipeline": "distilled"
    },
    "guidance_phases": 1,
    "num_inference_steps": 8,
    "video_length": 73,
    "resolution": "640x384"
}

# Align all default configs to Distilled INT8 weights
for config_file in ['ltx2_25_22B.json', 'ltx2_25_22B_distilled.json', 'ltx2_25_22B_distilled_nvfp4.json']:
    cfg_path = os.path.join(DEFAULTS_DIR, config_file)
    try:
        with open(cfg_path, 'w', encoding='utf-8') as f:
            json.dump(distilled_default_config, f, indent=4)
    except Exception as e:
        print(f'  Note writing {config_file}: {e}')

# Clean any remaining cache directories to maximize free disk & RAM
for d in ['/kaggle/working/.cache', os.path.join(MODEL_DIR, '.cache'), os.path.join(TMP_DIR, '.cache'), os.path.join(GEMMA4_DIR, '.cache')]:
    if os.path.exists(d):
        shutil.rmtree(d, ignore_errors=True)

print('\n✅ Official LTX-2.5 22B Distilled weights & environment prepared successfully!')

In [ ]:
#@title Step 3: Write the AIQUEST Academy High-Speed Streaming Web Application
import os
import sys

aiquest_app_code = """import gc
import os
import sys
import json
import time
import random
import tempfile
import glob
import traceback
import queue
import threading
import numpy as np
import subprocess
import psutil
import soundfile as sf
import torch
from PIL import Image

# ── Bootstrap Wan2GP ──
WAN2GP_DIR = os.path.abspath("Wan2GP")
if WAN2GP_DIR not in sys.path:
    sys.path.insert(0, WAN2GP_DIR)
os.chdir(WAN2GP_DIR)

OUTPUTS_DIR = os.path.abspath(os.path.join(WAN2GP_DIR, "outputs"))
os.makedirs(OUTPUTS_DIR, exist_ok=True)
os.makedirs("/kaggle/working/outputs", exist_ok=True)

# Enable PyTorch & cuDNN fast inference optimizations
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

# Clean any leftover lock files from previous runs
for lock_name in ['startup.lock', 'wgp.lock']:
    lock_path = os.path.join(WAN2GP_DIR, lock_name)
    if os.path.exists(lock_path):
        try:
            os.remove(lock_path)
        except Exception:
            pass

# Backward-compatibility patch for HfFolder in huggingface_hub
try:
    import huggingface_hub.utils as hf_utils
    if not hasattr(hf_utils, 'HfFolder'):
        class HfFolder:
            @staticmethod
            def get_token():
                import os
                try:
                    from huggingface_hub import get_token
                    return get_token() or os.environ.get('HF_TOKEN')
                except Exception:
                    return os.environ.get('HF_TOKEN')
            @staticmethod
            def save_token(token):
                pass
        setattr(hf_utils, 'HfFolder', HfFolder)
        if 'huggingface_hub.utils' in sys.modules:
            setattr(sys.modules['huggingface_hub.utils'], 'HfFolder', HfFolder)
except Exception:
    pass

# Patch ltx2.py in Wan2GP to disable requires_grad on empty models so int8 weights load cleanly
ltx2_file = os.path.join(WAN2GP_DIR, "models", "ltx2", "ltx2.py")
if os.path.isfile(ltx2_file):
    with open(ltx2_file, "r", encoding="utf-8") as f:
        ltx2_code = f.read()
    if 'def _load_component(model, path, sd_ops=None, postprocess=None, ignore_unused_weights=False):\\n            if postprocess is None and sd_ops is not None:' in ltx2_code:
        ltx2_code = ltx2_code.replace(
            'def _load_component(model, path, sd_ops=None, postprocess=None, ignore_unused_weights=False):\\n            if postprocess is None and sd_ops is not None:',
            'def _load_component(model, path, sd_ops=None, postprocess=None, ignore_unused_weights=False):\\n            model.requires_grad_(False)\\n            if postprocess is None and sd_ops is not None:'
        )
        with open(ltx2_file, "w", encoding="utf-8") as f:
            f.write(ltx2_code)

# Patch distilled.py in Wan2GP to use model dtype instead of hardcoded bfloat16 (enables fast FP16 on T4)
distilled_file = os.path.join(WAN2GP_DIR, "models", "ltx2", "ltx_pipelines", "distilled.py")
if os.path.isfile(distilled_file):
    with open(distilled_file, "r", encoding="utf-8") as f:
        dist_code = f.read()
    dist_code = dist_code.replace("self.dtype = torch.bfloat16", "self.dtype = getattr(models, 'dtype', torch.float16)")
    dist_code = dist_code.replace("dtype = torch.bfloat16", "dtype = getattr(self, 'dtype', torch.float16)")
    with open(distilled_file, "w", encoding="utf-8") as f:
        f.write(dist_code)

# Patch CausalConv3d in Wan2GP video_vae to prevent dtype mismatch between latents and VAE conv weights
conv_file = os.path.join(WAN2GP_DIR, "models", "ltx2", "ltx_core", "model", "video_vae", "convolution.py")
if os.path.isfile(conv_file):
    with open(conv_file, "r", encoding="utf-8") as f:
        conv_code = f.read()
    if "x = self.conv(x)" in conv_code and "x.dtype != self.conv.weight.dtype" not in conv_code:
        conv_code = conv_code.replace(
            "        x = self.conv(x)",
            "        if hasattr(self, 'conv') and hasattr(self.conv, 'weight') and x.dtype != self.conv.weight.dtype:\\n            x = x.to(dtype=self.conv.weight.dtype)\\n        x = self.conv(x)"
        )
        with open(conv_file, "w", encoding="utf-8") as f:
            f.write(conv_code)

# Patch audio_vae.py in Wan2GP to auto-cast latents to audio decoder and vocoder weight dtypes
audio_vae_file = os.path.join(WAN2GP_DIR, "models", "ltx2", "ltx_core", "model", "audio_vae", "audio_vae.py")
if os.path.isfile(audio_vae_file):
    with open(audio_vae_file, "r", encoding="utf-8") as f:
        audio_vae_code = f.read()
    if "decoded_audio = audio_decoder(latent)" in audio_vae_code and "latent.to(dtype=" not in audio_vae_code:
        audio_vae_code = audio_vae_code.replace(
            "    decoded_audio = audio_decoder(latent)",
            "    if hasattr(audio_decoder, 'conv_in') and hasattr(audio_decoder.conv_in, 'weight'):\\n        latent = latent.to(dtype=audio_decoder.conv_in.weight.dtype)\\n    decoded_audio = audio_decoder(latent)\\n    if hasattr(vocoder, 'vocoder') and hasattr(vocoder.vocoder, 'conv_pre'):\\n        decoded_audio = decoded_audio.to(dtype=vocoder.vocoder.conv_pre.weight.dtype)"
        )
        with open(audio_vae_file, "w", encoding="utf-8") as f:
            f.write(audio_vae_code)

# Patch vocoder.py in Wan2GP to prevent buffer dtype mismatch in LowPassFilter1d and STFT
vocoder_file = os.path.join(WAN2GP_DIR, "models", "ltx2", "ltx_core", "model", "audio_vae", "vocoder.py")
if os.path.isfile(vocoder_file):
    with open(vocoder_file, "r", encoding="utf-8") as f:
        vocoder_code = f.read()
    vocoder_code = vocoder_code.replace(
        "return F.conv1d(x, self.filter.expand(n_channels, -1, -1), stride=self.stride, groups=n_channels)",
        "filt = self.filter.to(dtype=x.dtype, device=x.device).expand(n_channels, -1, -1)\\n        return F.conv1d(x, filt, stride=self.stride, groups=n_channels)"
    )
    vocoder_code = vocoder_code.replace(
        "spec = F.conv1d(y, self.forward_basis, stride=self.hop_length, padding=0)",
        "basis = self.forward_basis.to(dtype=y.dtype, device=y.device)\\n        spec = F.conv1d(y, basis, stride=self.hop_length, padding=0)"
    )
    with open(vocoder_file, "w", encoding="utf-8") as f:
        f.write(vocoder_code)

# Patch PixelNorm in Wan2GP to compute RMS in float32 (fixes FP16 overflow into flat gray background)
norm_file = os.path.join(WAN2GP_DIR, "models", "ltx2", "ltx_core", "model", "common", "normalization.py")
if os.path.isfile(norm_file):
    with open(norm_file, "r", encoding="utf-8") as f:
        norm_code = f.read()
    if "x_float = x.float()" not in norm_code:
        norm_code = norm_code.replace(
            "        mean_sq = torch.mean(x**2, dim=self.dim, keepdim=True)\\n        # Normalize by the root-mean-square (RMS).\\n        rms = torch.sqrt(mean_sq + self.eps)\\n        return x / rms",
            "        orig_dtype = x.dtype\\n        x_float = x.float()\\n        mean_sq = torch.mean(x_float**2, dim=self.dim, keepdim=True)\\n        rms = torch.sqrt(mean_sq + self.eps)\\n        return (x_float / rms).to(orig_dtype)"
        )
        with open(norm_file, "w", encoding="utf-8") as f:
            f.write(norm_code)

# Patch feature_extractor.py in Wan2GP to compute token RMS in float32
fe_file = os.path.join(WAN2GP_DIR, "models", "ltx2", "ltx_core", "text_encoders", "gemma", "feature_extractor.py")
if os.path.isfile(fe_file):
    with open(fe_file, "r", encoding="utf-8") as f:
        fe_code = f.read()
    if "enc_f = encoded_text.float()" not in fe_code:
        fe_code = fe_code.replace(
            "def _norm_and_concat_per_token_rms(encoded_text: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\\n    b, t, d, l = encoded_text.shape\\n    variance = torch.mean(encoded_text**2, dim=2, keepdim=True)\\n    normed = encoded_text * torch.rsqrt(variance + 1e-6)\\n    normed = normed.reshape(b, t, d * l)\\n    mask_3d = attention_mask.bool().unsqueeze(-1)\\n    return torch.where(mask_3d, normed, torch.zeros_like(normed))",
            "def _norm_and_concat_per_token_rms(encoded_text: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\\n    orig_dtype = encoded_text.dtype\\n    enc_f = encoded_text.float()\\n    b, t, d, l = enc_f.shape\\n    variance = torch.mean(enc_f**2, dim=2, keepdim=True)\\n    normed = enc_f * torch.rsqrt(variance + 1e-6)\\n    normed = normed.reshape(b, t, d * l).to(orig_dtype)\\n    mask_3d = attention_mask.bool().unsqueeze(-1)\\n    return torch.where(mask_3d, normed, torch.zeros_like(normed))"
        )
        with open(fe_file, "w", encoding="utf-8") as f:
            f.write(fe_code)

import gradio as gr
from shared.utils.audio_video import save_video
from mmgp import offload, quant_router
from models.ltx2.ltx2_handler import family_handler

# Register Wan2GP custom quantization handlers (INT8 ConvRot, FP8, NVFP4, GGUF) with mmgp
_HANDLER_MODULES = [
    "shared.qtypes.scaled_fp8",
    "shared.qtypes.nvfp4",
    "shared.qtypes.bnb_nf4",
    "shared.qtypes.nunchaku_int4",
    "shared.qtypes.nunchaku_fp4",
    "shared.qtypes.asym_w4a8_int8",
    "shared.qtypes.int8_convrot",
    "shared.qtypes.gguf",
]
quant_router.unregister_handler(".fp8_quanto_bridge")
for handler in _HANDLER_MODULES:
    quant_router.register_handler(handler)
from shared.qtypes import gguf as gguf_handler
quant_router.register_file_extension("gguf", gguf_handler)

# ==== GPU INFO ====
gpu_count = torch.cuda.device_count()
print(f"GPUs Available: {gpu_count}")
for i in range(gpu_count):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB VRAM)")

ram = psutil.virtual_memory()
print(f"RAM: {ram.total / 1024**3:.1f} GB total, {ram.available / 1024**3:.1f} GB available")
sys.stdout.flush()

# ==== LOAD LTX-2.5 22B DISTILLED MODEL IN-MEMORY (HIGH-SPEED BUDGET ARCHITECTURE) ====
print("\\n=== Loading LTX-2.5 22B Distilled Model in Memory (One-Time Startup) ===")
sys.stdout.flush()

base_model_type = "ltx2_25_22B"
model_def = {
    "ltx2_pipeline": "distilled",
    "name": "LTX-2 2.5 Distilled 22B",
    "architecture": "ltx2_25_22B"
}
extra = family_handler.query_model_def(base_model_type, model_def)
model_def.update(extra)

# Auto-resolve and verify text encoder and transformer paths
for base_path in ["models", "ckpts"]:
    nested_gemma = os.path.join(base_path, "gemma4-12b-ltx-v1", "gemma4-12b-ltx-v1")
    if os.path.isdir(nested_gemma):
        for f in os.listdir(nested_gemma):
            dst = os.path.join(base_path, "gemma4-12b-ltx-v1", f)
            if not os.path.exists(dst):
                try:
                    os.replace(os.path.join(nested_gemma, f), dst)
                except Exception:
                    pass

# Ensure all symlinks exist between /tmp/models, models/, and ckpts/
for d_from in ["/tmp/models", "models"]:
    if os.path.exists(d_from):
        for f in os.listdir(d_from):
            if f.startswith("."): continue
            src = os.path.join(d_from, f)
            for d_to in ["models", "ckpts"]:
                dst = os.path.join(d_to, f)
                if not os.path.exists(dst):
                    try:
                        os.symlink(os.path.abspath(src), dst)
                    except Exception:
                        pass

# Specifically link gemma4-12b-ltx-v1 folder contents
for gf in ["gemma4-12b-ltx-v1_int8_convrot.safetensors", "config.json", "chat_template.jinja", "tokenizer.json", "tokenizer_config.json"]:
    src_gm = os.path.join("models", "gemma4-12b-ltx-v1", gf)
    dst_gc = os.path.join("ckpts", "gemma4-12b-ltx-v1", gf)
    os.makedirs(os.path.dirname(dst_gc), exist_ok=True)
    if os.path.exists(src_gm) and not os.path.exists(dst_gc):
        try:
            os.symlink(os.path.abspath(src_gm), dst_gc)
        except Exception:
            pass

text_encoder_file = "ckpts/gemma4-12b-ltx-v1/gemma4-12b-ltx-v1_int8_convrot.safetensors"
if not os.path.exists(text_encoder_file):
    text_encoder_file = "models/gemma4-12b-ltx-v1/gemma4-12b-ltx-v1_int8_convrot.safetensors"

transformer_path = "models/ltx-2.5-22b-distilled_diffusion_model_int8_convrot.safetensors"
if not os.path.exists(transformer_path):
    transformer_path = "ckpts/ltx-2.5-22b-distilled_diffusion_model_int8_convrot.safetensors"
if not os.path.exists(transformer_path) and os.path.exists("/tmp/models/ltx-2.5-22b-distilled_diffusion_model_int8_convrot.safetensors"):
    transformer_path = "/tmp/models/ltx-2.5-22b-distilled_diffusion_model_int8_convrot.safetensors"

print(f"  Transformer : {os.path.basename(transformer_path)}")
print(f"  Text Encoder: {os.path.basename(text_encoder_file)}")
sys.stdout.flush()

MODEL_DTYPE = torch.float16
VAE_DTYPE   = torch.float16

with torch.inference_mode():
    with torch.set_grad_enabled(False):
        wan_model, pipe = family_handler.load_model(
            model_filename=transformer_path,
            model_type="ltx2_25_22B_distilled",
            base_model_type=base_model_type,
            model_def=model_def,
            dtype=MODEL_DTYPE,
            VAE_dtype=VAE_DTYPE,
            text_encoder_filename=text_encoder_file,
        )

kwargs = {}
if isinstance(pipe, dict) and "pipe" in pipe:
    kwargs = pipe
    pipe = kwargs.pop("pipe")

loras = kwargs.pop("loras", [])

if "transformer" in pipe and "transformer" not in loras:
    loras.append("transformer")

print("\\nApplying mmgp Profile 4 with Partial Pinning & Free RAM Headroom (6000 MB transformer budget, Async DMA Transfers)...")
sys.stdout.flush()

offload.profile(
    pipe,
    profile_no=4,
    pinnedMemory="transformer",
    partialPinning=True,
    perc_reserved_mem_max=0.35,
    asyncTransfers=True,
    quantizeTransformer=False,
    convertWeightsFloatTo=torch.float16,
    loras=loras,
    budgets={
        # 6000 MB transformer budget keeps ~8.5 GB VRAM free for zero-thrashing SDPA attention
        "transformer":       6000,
        "text_encoder":      1500,
        "video_encoder":     2000,
        "video_decoder":     3000,
        "audio_encoder":     1000,
        "audio_decoder":     1000,
        "vocoder":           500,
        "spatial_upsampler": 1500,
        "vae":               1000,
        "*":                 1000,
    },
    **kwargs
)

offload.shared_state["_attention"] = "sdpa"
offload.shared_state["_radial"] = False

print("\\n✅ Setup complete! LTX-2.5 22B Distilled Model loaded & pinned in memory permanently.")
sys.stdout.flush()

# ==== RESOLUTION PRESETS ====
def get_resolution(preset_str, aspect_ratio_str):
    base_resolutions = {
        "Fast Preview (384p - ~1-2 min)": 384,
        "Balanced (480p - ~3-5 min)": 480,
        "High Quality (704p - ~6-8 min)": 704,
        "Cinema 1080p (1088p - High Detail)": 1088,
    }
    ratios = {
        "16:9 Landscape": 16/9, "4:3 Standard": 4/3,
        "1:1 Square": 1.0, "3:4 Portrait": 3/4, "9:16 Portrait": 9/16,
    }
    base = base_resolutions.get(preset_str, 384)
    ratio = ratios.get(aspect_ratio_str, 16/9)
    if ratio >= 1.0:
        height = base
        width = int(base * ratio)
    else:
        width = base
        height = int(base / ratio)
    return (width // 32) * 32, (height // 32) * 32

# ==== REAL-TIME STREAMING VIDEO GENERATION (TIMEOUT-FREE) ====
def Video_Generation(prompt, input_image_start, input_image_end, seed, duration_dropdown,
                     resolution_dropdown, aspect_ratio_dropdown,
                     guide_scale=3.0, audio_cfg=7.0, num_steps=8):
    try:
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
        offload.shared_state["_attention"] = "sdpa"
        offload.shared_state["_radial"] = False

        duration_map = {
            "2 Seconds (49 frames - Fast)": 49,
            "3 Seconds (73 frames - Standard)": 73,
            "5 Seconds (121 frames - Long)": 121,
            "8 Seconds (193 frames)": 193,
            "10 Seconds (241 frames)": 241,
        }
        frame_rate = 24.0
        num_frames = duration_map.get(duration_dropdown, 73)
        width, height = get_resolution(resolution_dropdown, aspect_ratio_dropdown)

        if seed is None or seed < 0:
            seed = random.randint(0, 2**32 - 1)
        seed = int(seed)

        image_start = None
        image_end   = None
        if input_image_start is not None:
            image_start = Image.open(input_image_start).convert("RGB")
        if input_image_end is not None:
            image_end = Image.open(input_image_end).convert("RGB")

        free_vram = torch.cuda.mem_get_info()[0] / 1024**3
        ram = psutil.virtual_memory()
        mode = "T2V" if image_start is None else ("I2V first+last" if image_end else "I2V start")

        print(f"\\n{'='*60}")
        print(f"🎬 Generating LTX-2.5 [{mode}]: {width}x{height}, {num_frames} frames (~{num_frames/24.0:.1f}s), seed={seed}")
        print(f"  VRAM free: {free_vram:.2f} GB | RAM free: {ram.available / 1024**3:.1f} GB")
        print(f"  Prompt: {prompt[:120]}{'...' if len(prompt) > 120 else ''}")
        print(f"  Video CFG: {guide_scale} | Audio CFG: {audio_cfg} | Steps: {num_steps}")
        print(f"{'='*60}")
        sys.stdout.flush()

        yield None, f"⏳ Initializing LTX-2.5 [{mode}] ({width}x{height}, {num_frames} frames, seed: {seed})..."

        # Progress queue for real-time WebSocket updates across threads
        msg_queue = queue.Queue()
        total_steps = [int(num_steps)]
        current_step = [0]
        step_times = []
        last_step_time = [time.time()]

        def cb(step, latent, is_start, override_num_inference_steps=None, pass_no=None, **kwargs):
            if is_start:
                if override_num_inference_steps is not None:
                    total_steps[0] = override_num_inference_steps
                current_step[0] = 0
                last_step_time[0] = time.time()
                return
            now = time.time()
            dt = now - last_step_time[0]
            last_step_time[0] = now
            step_times.append(dt)
            current_step[0] += 1
            free_v = torch.cuda.mem_get_info()[0] / 1024**3
            avg_dt = sum(step_times) / len(step_times)
            rem_steps = max(total_steps[0] - current_step[0], 0)
            rem_sec = rem_steps * avg_dt
            msg = f"⚡ Denoising step {current_step[0]}/{total_steps[0]} ({dt:.1f}s/step | ~{rem_sec:.0f}s left | VRAM: {free_v:.1f} GB free)"
            print(f"  [Denoising] {msg}")
            sys.stdout.flush()
            msg_queue.put(msg)

        _stage_labels = {
            "VAE Encoding": "🎞️ VAE Encoding input frames...",
            "VAE Decoding": "🎬 VAE Decoding latents → video frames...",
            "Upsampling":   "🔭 Spatial upsampling latents...",
        }

        def set_progress_status(status: str):
            label = _stage_labels.get(status, f"⏳ {status}...")
            print(f"  [{status}] {label}")
            sys.stdout.flush()
            msg_queue.put(label)
            if "Decoding" in status or "Upsampling" in status:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        gen_kwargs = dict(
            input_prompt=prompt,
            image_start=image_start,
            height=height,
            width=width,
            frame_num=num_frames,
            fps=frame_rate,
            seed=seed,
            callback=cb,
            VAE_tile_size=0,
            input_video_strength=1.0,
            denoising_strength=1.0,
            guide_scale=float(guide_scale),
            audio_cfg_scale=float(audio_cfg),
            sampling_steps=int(num_steps),
            guide_phases=1,
            sample_solver="distilled_8_steps",
            self_refiner_setting=0,
            n_prompt="",
            set_progress_status=set_progress_status,
        )
        if image_end is not None:
            gen_kwargs["image_end"] = image_end

        result_holder = {}
        error_holder = {}

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        def run_generation():
            try:
                t0 = time.time()
                res = wan_model.generate(**gen_kwargs)
                result_holder["result"] = res
                result_holder["elapsed"] = time.time() - t0
            except Exception as exc:
                error_holder["error"] = exc
                traceback.print_exc()

        gen_thread = threading.Thread(target=run_generation)
        gen_thread.start()

        # Stream periodic WebSocket heartbeats while generation is in progress
        last_yield_time = time.time()
        current_status = "⏳ Starting in-memory single-stage 8-step generation..."
        while gen_thread.is_alive():
            # Check for new step messages
            updated = False
            while not msg_queue.empty():
                current_status = msg_queue.get_nowait()
                updated = True

            # Yield heartbeat every 2 seconds to keep WebSocket proxy alive
            now = time.time()
            if updated or (now - last_yield_time >= 2.0):
                last_yield_time = now
                yield None, current_status
            time.sleep(0.5)

        gen_thread.join()

        if "error" in error_holder:
            raise error_holder["error"]

        result = result_holder.get("result")
        elapsed = result_holder.get("elapsed", 0)
        print(f"  [Pipeline Completed] Finished in {elapsed:.1f}s")
        sys.stdout.flush()

        yield None, f"🎬 Generation finished in {elapsed:.1f}s! Saving MP4 and muxing audio..."

        if result is None:
            yield None, "❌ Generation returned no output."
            return

        audio_data = None
        audio_sr = 24000
        if isinstance(result, dict):
            video_tensor = result.get("x")
            audio_data   = result.get("audio")
            audio_sr     = result.get("audio_sampling_rate", 24000)
        elif isinstance(result, tuple):
            video_tensor = result[0]
            audio_data   = result[1] if len(result) > 1 else None
            audio_sr     = result[2] if len(result) > 2 else 24000
        else:
            video_tensor = result
            audio_data, audio_sr = None, 24000

        if video_tensor is None or not torch.is_tensor(video_tensor):
            yield None, f"❌ No video tensor returned. Got: {type(video_tensor)}"
            return

        video_tensor = video_tensor.cpu()
        gc.collect(); torch.cuda.empty_cache()

        # Save video directly to physical disk OUTPUTS_DIR (not tmpfs RAM)
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        out_filename = f"ltx25_{timestamp}_seed{seed}.mp4"
        out_path = os.path.join(OUTPUTS_DIR, out_filename)

        if video_tensor.dtype != torch.uint8:
            video_tensor = video_tensor.clamp(0, 255).to(torch.uint8)
        if video_tensor.ndim == 4:
            video_tensor = video_tensor.unsqueeze(0)
        save_video(tensor=video_tensor, save_file=out_path, fps=frame_rate, nrow=1)

        # Also copy to /kaggle/working/outputs for external access
        try:
            import shutil
            shutil.copy2(out_path, os.path.join("/kaggle/working/outputs", out_filename))
        except Exception:
            pass

        # ==== Mux native synchronized audio (if generated) ====
        if audio_data is not None:
            try:
                audio_tmp = tempfile.mktemp(suffix=".wav")
                if isinstance(audio_data, np.ndarray):
                    audio_np = audio_data
                    if audio_np.ndim == 2 and audio_np.shape[0] <= 2 and audio_np.shape[1] > audio_np.shape[0]:
                        audio_np = audio_np.T
                    sf.write(audio_tmp, audio_np, int(audio_sr or 24000))
                elif torch.is_tensor(audio_data):
                    import torchaudio
                    cpu_audio = audio_data.cpu().float()
                    if cpu_audio.dim() == 1: cpu_audio = cpu_audio.unsqueeze(0)
                    if cpu_audio.dim() == 3: cpu_audio = cpu_audio.squeeze(0)
                    torchaudio.save(audio_tmp, cpu_audio, int(audio_sr or 24000))

                final_path = out_path.replace(".mp4", "_with_audio.mp4")
                subprocess.run([
                    "ffmpeg", "-y", "-i", out_path, "-i", audio_tmp,
                    "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
                    "-shortest", final_path
                ], check=True, capture_output=True)
                if os.path.exists(final_path) and os.path.getsize(final_path) > 0:
                    out_path = final_path
                    # Copy audio-muxed version to persistent storage
                    try:
                        shutil.copy2(out_path, os.path.join("/kaggle/working/outputs", os.path.basename(final_path)))
                        shutil.copy2(out_path, os.path.join(OUTPUTS_DIR, os.path.basename(final_path)))
                    except Exception:
                        pass
                    print(f"  ✅ Synchronized audio muxed into output: {out_path}")
            except Exception as e:
                print(f"  ⚠️ Audio mux note: {e}")

        del video_tensor
        gc.collect(); torch.cuda.empty_cache()

        yield out_path, f"✅ Video generated in {elapsed:.1f}s! Seed: {seed} | {width}x{height} | {num_frames} frames | Saved: {out_path}"

    except Exception as e:
        traceback.print_exc()
        gc.collect(); torch.cuda.empty_cache()
        yield None, f"❌ Error: {str(e)}"

# ==== GRADIO UI (AIQUEST BRANDED) ====
CSS = '''@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1050px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.social-buttons { display: flex; justify-content: center; gap: 12px; flex-wrap: wrap; }
.social-btn { padding: 10px 24px; border-radius: 8px; font-weight: 700; font-size: 15px; text-decoration: none; display: inline-block; color: white !important; transition: all 0.3s; box-shadow: 0 4px 12px rgba(0,0,0,0.2); }
.social-btn:hover { transform: translateY(-2px); box-shadow: 0 6px 16px rgba(0,0,0,0.3); }
.youtube-btn { background: linear-gradient(135deg, #FF0000 0%, #CC0000 100%); }
.x-btn { background: linear-gradient(135deg, #000000 0%, #333333 100%); }
button.primary, #gen-btn { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#stop-btn { background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#clear-btn { background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
.footer { text-align: center; padding: 20px; margin-top: 30px; border-top: 2px solid #e5e7eb; color: #6b7280; }
'''

with gr.Blocks(css=CSS, theme=gr.themes.Soft(), title="LTX-2.5 22B Distilled - AIQUEST Academy") as demo:
    gr.HTML('<div class="brand-header"><div class="brand-title">🎬 LTX-2.5 22B Distilled - Video Generator</div><div class="brand-subtitle">Created by <strong>AIQUEST Academy</strong> | Kaggle Dual T4 GPU Edition</div><div class="social-buttons"><a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn youtube-btn">▶️ Subscribe on YouTube</a><a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a></div></div>')

    gr.Markdown(
        '''**Single-Stage 8-Step Distilled Pipeline** | Official INT8 Convrot (~19.5 GB) | Persistent In-Memory Model

⚡ **Performance Presets:** Use **Fast Preview (384p, 2s/3s)** for rapid ~1-2 minute generation. All generated videos are automatically saved to `/kaggle/working/outputs/`.'''
    )

    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(
                label="🎨 Prompt", lines=3,
                placeholder="A cinematic 4k close-up shot of a majestic snow leopard in the Himalayas during a snowfall, breathing softly, ultra realistic..."
            )

            with gr.Accordion("🖼️ Image to Video (Optional)", open=False):
                with gr.Row():
                    input_image_start = gr.Image(type="filepath", label="🎬 Start Frame (First Frame)", height=180)
                    input_image_end   = gr.Image(type="filepath", label="🎬 End Frame (Last Frame - Optional)", height=180)
                gr.Markdown(
                    '''*• **Start Frame only** → Image-to-Video (model generates from your image)*
*• **Both frames** → First+Last frame interpolation (model generates in-between)*
*• **Neither** → Pure Text-to-Video*'''
                )

            with gr.Row():
                duration_dropdown = gr.Dropdown(
                    label="⏱️ Duration",
                    choices=[
                        "2 Seconds (49 frames - Fast)",
                        "3 Seconds (73 frames - Standard)",
                        "5 Seconds (121 frames - Long)",
                        "8 Seconds (193 frames)",
                        "10 Seconds (241 frames)",
                    ],
                    value="5 Seconds (121 frames - Long)",
                )
                resolution_dropdown = gr.Dropdown(
                    label="📐 Resolution Preset",
                    choices=[
                        "Fast Preview (384p - ~1-2 min)",
                        "Balanced (480p - ~3-5 min)",
                        "High Quality (704p - ~6-8 min)",
                        "Cinema 1080p (1088p - High Detail)",
                    ],
                    value="Balanced (480p - ~3-5 min)",
                )
                aspect_ratio_dropdown = gr.Dropdown(
                    label="📏 Aspect Ratio",
                    choices=["16:9 Landscape", "4:3 Standard", "1:1 Square", "3:4 Portrait", "9:16 Portrait"],
                    value="16:9 Landscape",
                )

            with gr.Row():
                guide_scale = gr.Slider(
                    label="🎯 Video CFG Scale (Distilled: 1.0 recommended)",
                    minimum=1.0, maximum=3.0, step=0.1, value=1.0,
                )
                audio_cfg = gr.Slider(
                    label="🔊 Audio CFG Scale",
                    minimum=1.0, maximum=5.0, step=0.5, value=1.0,
                )

            with gr.Row():
                num_steps = gr.Slider(
                    label="⚡ Denoising Steps",
                    minimum=4, maximum=12, step=1, value=8,
                )
                seed = gr.Number(label="🎲 Seed (-1 for Random)", value=-1, precision=0)

            with gr.Row():
                gen_btn   = gr.Button("🎬 Generate Video", variant="primary", size="lg", elem_id="gen-btn")
                stop_btn  = gr.Button("🛑 Stop",            variant="secondary", size="lg", elem_id="stop-btn")
                clear_btn = gr.Button("🗑️ Clear",           variant="secondary", size="lg", elem_id="clear-btn")

        with gr.Column(scale=1):
            video_out  = gr.Video(label="🎥 Generated Video with Synchronized Audio", height=420)
            status_out = gr.Textbox(label="ℹ️ Live Progress & Output Path", interactive=False, lines=2)

    gr.HTML('<div class="footer"><p style="font-size: 16px; margin: 5px 0; text-align: center;">🎬 Created by <strong>AIQUEST Academy</strong></p><p style="font-size: 14px; margin: 5px 0; color: #9ca3af; text-align: center;">Official LTX-2.5 22B Distilled INT8 Convrot | Kaggle Dual GPU (GPU T4 x2)</p><p style="font-size: 13px; margin: 10px 0; text-align: center;"><a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px;">YouTube</a> • <a href="https://x.com/aiquestacademy" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px;">X (Twitter)</a></p></div>')

    gen_event = gen_btn.click(
        fn=Video_Generation,
        inputs=[prompt, input_image_start, input_image_end, seed, duration_dropdown,
                resolution_dropdown, aspect_ratio_dropdown, guide_scale, audio_cfg, num_steps],
        outputs=[video_out, status_out],
    )
    stop_btn.click(fn=None, cancels=[gen_event])
    clear_btn.click(
        fn=lambda: (None, None, None, "", -1),
        outputs=[input_image_start, input_image_end, video_out, prompt, seed],
    )

print("\\n🚀 Launching AIQUEST Academy Gradio Web Interface...")
sys.stdout.flush()
demo.queue(max_size=20, default_concurrency_limit=1)

allowed_paths = [
    os.path.abspath("outputs"),
    os.path.abspath(WAN2GP_DIR),
    "/kaggle/working",
    "/kaggle/working/outputs",
    "/kaggle/working/Wan2GP",
    "/tmp",
    tempfile.gettempdir()
]

demo.launch(
    share=True,
    inline=False,
    debug=True,
    show_error=True,
    max_threads=4,
    ssr_mode=False,
    allowed_paths=allowed_paths
)
"""

with open("run_aiquest_ltx25.py", "w", encoding="utf-8") as f:
    f.write(aiquest_app_code)

print("✅ run_aiquest_ltx25.py created successfully!")

In [ ]:
#@title Step 3.5: Hotfix - High-Speed FP16 + INT8 ConvRot + Full Audio/Video VAE Fix (Run THIS then Step 4)
import os, re

# ============================================================
# CRITICAL FIX 1: Patch distilled.py to run in FP16 on T4 (24s/step)
# ============================================================
for base_dir in ["Wan2GP", "."]:
    distilled_file = os.path.join(base_dir, "models", "ltx2", "ltx_pipelines", "distilled.py")
    if os.path.isfile(distilled_file):
        with open(distilled_file, "r", encoding="utf-8") as f:
            dist_code = f.read()
        dist_code = dist_code.replace("self.dtype = torch.bfloat16", "self.dtype = getattr(models, 'dtype', torch.float16)")
        dist_code = dist_code.replace("dtype = torch.bfloat16", "dtype = getattr(self, 'dtype', torch.float16)")
        with open(distilled_file, "w", encoding="utf-8") as f:
            f.write(dist_code)
        print(f"✅ Patched distilled.py for high-speed FP16 in {distilled_file}")

# ============================================================
# CRITICAL FIX 2: Patch CausalConv3d in Wan2GP video_vae on disk
# Prevents any BFloat16 vs Half dtype mismatch in VAE 3D conv layers
# ============================================================
for base_dir in ["Wan2GP", "."]:
    conv_file = os.path.join(base_dir, "models", "ltx2", "ltx_core", "model", "video_vae", "convolution.py")
    if os.path.isfile(conv_file):
        with open(conv_file, "r", encoding="utf-8") as f:
            conv_code = f.read()
        if "x = self.conv(x)" in conv_code and "x.dtype != self.conv.weight.dtype" not in conv_code:
            conv_code = conv_code.replace(
                "        x = self.conv(x)",
                "        if hasattr(self, 'conv') and hasattr(self.conv, 'weight') and x.dtype != self.conv.weight.dtype:\n            x = x.to(dtype=self.conv.weight.dtype)\n        x = self.conv(x)"
            )
            with open(conv_file, "w", encoding="utf-8") as f:
                f.write(conv_code)
            print(f"✅ Patched VAE CausalConv3d auto-cast in {conv_file}")

# ============================================================
# CRITICAL FIX 3: Patch audio_vae.py & vocoder.py in Wan2GP on disk
# Prevents any buffer/weight dtype mismatch during audio decoding
# ============================================================
for base_dir in ["Wan2GP", "."]:
    audio_vae_file = os.path.join(base_dir, "models", "ltx2", "ltx_core", "model", "audio_vae", "audio_vae.py")
    if os.path.isfile(audio_vae_file):
        with open(audio_vae_file, "r", encoding="utf-8") as f:
            audio_vae_code = f.read()
        if "decoded_audio = audio_decoder(latent)" in audio_vae_code and "latent.to(dtype=" not in audio_vae_code:
            audio_vae_code = audio_vae_code.replace(
                "    decoded_audio = audio_decoder(latent)",
                "    if hasattr(audio_decoder, 'conv_in') and hasattr(audio_decoder.conv_in, 'weight'):\n        latent = latent.to(dtype=audio_decoder.conv_in.weight.dtype)\n    decoded_audio = audio_decoder(latent)\n    if hasattr(vocoder, 'vocoder') and hasattr(vocoder.vocoder, 'conv_pre'):\n        decoded_audio = decoded_audio.to(dtype=vocoder.vocoder.conv_pre.weight.dtype)"
            )
            with open(audio_vae_file, "w", encoding="utf-8") as f:
                f.write(audio_vae_code)
            print(f"✅ Patched audio_vae.py auto-cast in {audio_vae_file}")

    vocoder_file = os.path.join(base_dir, "models", "ltx2", "ltx_core", "model", "audio_vae", "vocoder.py")
    if os.path.isfile(vocoder_file):
        with open(vocoder_file, "r", encoding="utf-8") as f:
            vocoder_code = f.read()
        vocoder_code = vocoder_code.replace(
            "return F.conv1d(x, self.filter.expand(n_channels, -1, -1), stride=self.stride, groups=n_channels)",
            "filt = self.filter.to(dtype=x.dtype, device=x.device).expand(n_channels, -1, -1)\n        return F.conv1d(x, filt, stride=self.stride, groups=n_channels)"
        )
        vocoder_code = vocoder_code.replace(
            "spec = F.conv1d(y, self.forward_basis, stride=self.hop_length, padding=0)",
            "basis = self.forward_basis.to(dtype=y.dtype, device=y.device)\n        spec = F.conv1d(y, basis, stride=self.hop_length, padding=0)"
        )
        with open(vocoder_file, "w", encoding="utf-8") as f:
            f.write(vocoder_code)
        print(f"✅ Patched vocoder.py filter & STFT buffers in {vocoder_file}")

# ============================================================
# CRITICAL FIX 3.5: Patch PixelNorm & feature_extractor in FP32
# Prevents FP16 x**2 overflow (>65504 -> inf -> zero/gray background)
# ============================================================
for base_dir in ["Wan2GP", "."]:
    norm_file = os.path.join(base_dir, "models", "ltx2", "ltx_core", "model", "common", "normalization.py")
    if os.path.isfile(norm_file):
        with open(norm_file, "r", encoding="utf-8") as f:
            norm_code = f.read()
        if "x_float = x.float()" not in norm_code:
            norm_code = norm_code.replace(
                "        mean_sq = torch.mean(x**2, dim=self.dim, keepdim=True)\n        # Normalize by the root-mean-square (RMS).\n        rms = torch.sqrt(mean_sq + self.eps)\n        return x / rms",
                "        orig_dtype = x.dtype\n        x_float = x.float()\n        mean_sq = torch.mean(x_float**2, dim=self.dim, keepdim=True)\n        rms = torch.sqrt(mean_sq + self.eps)\n        return (x_float / rms).to(orig_dtype)"
            )
            with open(norm_file, "w", encoding="utf-8") as f:
                f.write(norm_code)
            print(f"✅ Patched PixelNorm float32 calculation (fixes gray background) in {norm_file}")

    fe_file = os.path.join(base_dir, "models", "ltx2", "ltx_core", "text_encoders", "gemma", "feature_extractor.py")
    if os.path.isfile(fe_file):
        with open(fe_file, "r", encoding="utf-8") as f:
            fe_code = f.read()
        if "enc_f = encoded_text.float()" not in fe_code:
            fe_code = fe_code.replace(
                "def _norm_and_concat_per_token_rms(encoded_text: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n    b, t, d, l = encoded_text.shape\n    variance = torch.mean(encoded_text**2, dim=2, keepdim=True)\n    normed = encoded_text * torch.rsqrt(variance + 1e-6)\n    normed = normed.reshape(b, t, d * l)\n    mask_3d = attention_mask.bool().unsqueeze(-1)\n    return torch.where(mask_3d, normed, torch.zeros_like(normed))",
                "def _norm_and_concat_per_token_rms(encoded_text: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n    orig_dtype = encoded_text.dtype\n    enc_f = encoded_text.float()\n    b, t, d, l = enc_f.shape\n    variance = torch.mean(enc_f**2, dim=2, keepdim=True)\n    normed = enc_f * torch.rsqrt(variance + 1e-6)\n    normed = normed.reshape(b, t, d * l).to(orig_dtype)\n    mask_3d = attention_mask.bool().unsqueeze(-1)\n    return torch.where(mask_3d, normed, torch.zeros_like(normed))"
            )
            with open(fe_file, "w", encoding="utf-8") as f:
                f.write(fe_code)
            print(f"✅ Patched feature_extractor float32 RMS precision in {fe_file}")

patch_file = "run_aiquest_ltx25.py"
with open(patch_file, "r", encoding="utf-8") as f:
    code = f.read()

# Remove any previously injected monkey-patches
if "# === AIQUEST HOTFIX" in code:
    code = re.sub(r'\n# === AIQUEST HOTFIX.*?# === END HOTFIX ===\n', '\n', code, flags=re.DOTALL)
    print("✅ Cleaned previous monkey-patch from script")

# ============================================================
# CRITICAL FIX 4: Register Wan2GP INT8 ConvRot handlers with mmgp
# ============================================================
quant_router_init = '''from mmgp import offload, quant_router
_HANDLER_MODULES = [
    "shared.qtypes.scaled_fp8",
    "shared.qtypes.nvfp4",
    "shared.qtypes.bnb_nf4",
    "shared.qtypes.nunchaku_int4",
    "shared.qtypes.nunchaku_fp4",
    "shared.qtypes.asym_w4a8_int8",
    "shared.qtypes.int8_convrot",
    "shared.qtypes.gguf",
]
quant_router.unregister_handler(".fp8_quanto_bridge")
for handler in _HANDLER_MODULES:
    quant_router.register_handler(handler)
from shared.qtypes import gguf as gguf_handler
quant_router.register_file_extension("gguf", gguf_handler)'''

if "quant_router.register_handler" not in code:
    code = code.replace("from mmgp import offload", quant_router_init)
    print("✅ Injected Wan2GP INT8 ConvRot quant_router registration")
else:
    print("✓ quant_router handlers already registered")

code = code.replace("MODEL_DTYPE = torch.bfloat16", "MODEL_DTYPE = torch.float16")
code = code.replace("VAE_DTYPE   = torch.bfloat16", "VAE_DTYPE   = torch.float16")
code = code.replace("convertWeightsFloatTo=torch.bfloat16", "convertWeightsFloatTo=torch.float16")
code = code.replace("profile_no=5,", "profile_no=4,\n    pinnedMemory=\"transformer\",\n    partialPinning=True,\n    perc_reserved_mem_max=0.35,\n    asyncTransfers=True,")
code = code.replace("pinnedMemory=False,", "pinnedMemory=\"transformer\",\n    partialPinning=True,\n    perc_reserved_mem_max=0.35,")
if "partialPinning" not in code:
    code = code.replace("pinnedMemory=\"transformer\",", "pinnedMemory=\"transformer\",\n    partialPinning=True,\n    perc_reserved_mem_max=0.35,")
if "convertWeightsFloatTo" not in code:
    code = code.replace(
        '    quantizeTransformer=False,',
        '    quantizeTransformer=False,\n    convertWeightsFloatTo=torch.float16,'
    )

# Fix 6: Set proper SDPA backends for T4 (Turing GPU uses mem_efficient_sdp)
code = code.replace(
    'torch.backends.cuda.enable_flash_sdp(True)',
    'torch.backends.cuda.enable_flash_sdp(False)\ntorch.backends.cuda.enable_mem_efficient_sdp(True)'
)
code = code.replace(
    'torch.backends.cuda.enable_flash_sdp(False)',
    'torch.backends.cuda.enable_flash_sdp(False)\ntorch.backends.cuda.enable_mem_efficient_sdp(True)'
)

# Fix 7: VAE_tile_size=0 (eliminates 256px tile seam boundary artifacts) & Solver="distilled_8_steps"
code = code.replace('VAE_tile_size=256,', 'VAE_tile_size=0,')
code = code.replace('sample_solver="distilled_8_steps_ancestral",', 'sample_solver="distilled_8_steps",')

# Fix 8: Save output to /tmp/ so Gradio can serve it (fixes InvalidPathError)
code = code.replace(
    '        # Save video to outputs directory\n'
    '        timestamp = time.strftime("%Y%m%d_%H%M%S")\n'
    '        out_filename = f"ltx25_{timestamp}_seed{seed}.mp4"\n'
    '        out_path = os.path.join(OUTPUTS_DIR, out_filename)',
    '        # Save video to /tmp/ for Gradio compatibility, then copy to persistent storage\n'
    '        timestamp = time.strftime("%Y%m%d_%H%M%S")\n'
    '        out_filename = f"ltx25_{timestamp}_seed{seed}.mp4"\n'
    '        TMP_OUTPUTS = os.path.join(tempfile.gettempdir(), "ltx25_outputs")\n'
    '        os.makedirs(TMP_OUTPUTS, exist_ok=True)\n'
    '        out_path = os.path.join(TMP_OUTPUTS, out_filename)'
)

# Fix 9: Copy to both persistent dirs
code = code.replace(
    '        # Also copy to /kaggle/working/outputs for external reference\n'
    '        try:\n'
    '            import shutil\n'
    '            shutil.copy2(out_path, os.path.join("/kaggle/working/outputs", out_filename))\n'
    '        except Exception:\n'
    '            pass',
    '        # Copy to persistent storage directories\n'
    '        try:\n'
    '            import shutil\n'
    '            shutil.copy2(out_path, os.path.join("/kaggle/working/outputs", out_filename))\n'
    '            shutil.copy2(out_path, os.path.join(OUTPUTS_DIR, out_filename))\n'
    '        except Exception:\n'
    '            pass'
)

# Fix 10: Also copy audio-muxed files to Wan2GP outputs
old_audio = '                    try:\n                        shutil.copy2(out_path, os.path.join("/kaggle/working/outputs", os.path.basename(final_path)))\n                    except Exception:\n                        pass'
new_audio = '                    try:\n                        shutil.copy2(out_path, os.path.join("/kaggle/working/outputs", os.path.basename(final_path)))\n                        shutil.copy2(out_path, os.path.join(OUTPUTS_DIR, os.path.basename(final_path)))\n                    except Exception:\n                        pass'
code = code.replace(old_audio, new_audio)

# Fix 11: Direct uint8 video saving (avoids per-frame make_grid normalize blowout)
code = code.replace(
    '        video_for_save = video_tensor.unsqueeze(0).float() / 127.5 - 1.0\n'
    '        save_video(tensor=video_for_save, save_file=out_path, fps=frame_rate, normalize=True, value_range=(-1, 1))',
    '        if video_tensor.dtype != torch.uint8:\n'
    '            video_tensor = video_tensor.clamp(0, 255).to(torch.uint8)\n'
    '        if video_tensor.ndim == 4:\n'
    '            video_tensor = video_tensor.unsqueeze(0)\n'
    '        save_video(tensor=video_tensor, save_file=out_path, fps=frame_rate, nrow=1)'
)
code = code.replace('del video_tensor, video_for_save', 'del video_tensor')

# Fix 12: Set Video CFG default to 1.0 (Distilled models require CFG=1.0) & Audio CFG default to 1.0
code = code.replace(
    'label="🎯 Video CFG Scale",\n                    minimum=1.0, maximum=7.0, step=0.5, value=3.0,',
    'label="🎯 Video CFG Scale (Distilled: 1.0 recommended)",\n                    minimum=1.0, maximum=3.0, step=0.1, value=1.0,'
)
code = code.replace(
    'label="🔊 Audio CFG Scale",\n                    minimum=1.0, maximum=10.0, step=0.5, value=7.0,',
    'label="🔊 Audio CFG Scale",\n                    minimum=1.0, maximum=5.0, step=0.5, value=1.0,'
)

# Fix 13: Default duration → 5 Seconds, Resolution → 480p
code = code.replace(
    'value="2 Seconds (49 frames - Fast)"',
    'value="5 Seconds (121 frames - Long)"'
)
code = code.replace(
    'value="3 Seconds (73 frames - Standard)"',
    'value="5 Seconds (121 frames - Long)"'
)
code = code.replace(
    'value="Fast Preview (384p - ~1-2 min)"',
    'value="Balanced (480p - ~3-5 min)"'
)

with open(patch_file, "w", encoding="utf-8") as f:
    f.write(code)

print("\n✅ All patches applied successfully to run_aiquest_ltx25.py!")
print("   → High-Speed FP16 enabled on T4 Tensor Cores (~24s/step)")
print("   → VAE_tile_size=0 (disabled spatial tiling seams)")
print("   → sample_solver=distilled_8_steps (clean deterministic Euler flow-matching)")
print("   → Video VAE & Audio Vocoder buffer auto-casts applied")
print("   → Video saving fixed (direct uint8 frames without make_grid distortion)")
print("   → Video CFG Scale set to 1.0 (eliminates blowout / posterization artifacts)")
print("   → Audio CFG Scale set to 1.0")
print("   → Default: 5 Seconds at 480p")
print("\n🔄 Now re-run Step 4 to launch the server.")

In [ ]:
#@title Step 4: Launch AIQUEST Academy LTX-2.5 22B Distilled Web Server (GPU T4 x2)
import os
import sys
import subprocess

print("=== Launching AIQUEST Academy LTX-2.5 22B Distilled Web Server ===")
print("Pre-loading model into memory and generating public Gradio link...")
sys.stdout.flush()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

# Run the standalone AIQUEST Academy web application
subprocess.run([sys.executable, "-u", "run_aiquest_ltx25.py"], env=env)